# Feature Engineering: SPECTER + Venue + Author

Replaces the TF-IDF text features with SPECTER2 embeddings.
Combines: SPECTER (768) + venue prestige (9) + author/collab (10) + metadata (8)

**Run `20c_specter_embeddings.ipynb` first** to generate `specter_features.pkl`.

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

feature_dir = Path('../data/features')
print("Imports OK")

## 1. Load All Feature Files

In [ ]:
specter_features = pd.read_pickle(feature_dir / 'specter_features.pkl')
venue_features   = pd.read_pickle(feature_dir / 'venue_features.pkl')
author_features  = pd.read_pickle(feature_dir / 'author_features.pkl')

# Optional extras
for fname, var_name in [('enhanced_features.pkl', 'enhanced'), ('additional_features.pkl', 'additional')]:
    try:
        globals()[var_name] = pd.read_pickle(feature_dir / fname)
        print(f"Loaded {fname}: {globals()[var_name].shape}")
    except FileNotFoundError:
        globals()[var_name] = None

print(f"\nSPECTER  features: {specter_features.shape}")
print(f"Venue    features: {venue_features.shape}")
print(f"Author   features: {author_features.shape}")

## 2. Combine Features

In [ ]:
feature_list = [specter_features, venue_features, author_features]
if enhanced is not None:
    feature_list.append(enhanced)
if additional is not None:
    feature_list.append(additional)

# Align on common index
common_idx = feature_list[0].index
for f in feature_list[1:]:
    common_idx = common_idx.intersection(f.index)
print(f"Common papers: {len(common_idx)}")

X = pd.concat([f.loc[common_idx] for f in feature_list], axis=1)

# Fill any NaN
X = X.fillna(X.median())

print(f"\nCombined feature matrix: {X.shape}")
print(f"  vs TF-IDF baseline: (N, 5019)")
print(f"  Feature reduction: {5019 - X.shape[1]:+d} columns (dense vs sparse)")

## 3. Load Target

In [ ]:
y = pd.read_pickle(feature_dir / 'y_classification.pkl').reindex(common_idx)
print(f"Target: {len(y)} papers, {y.sum()} high-impact ({y.mean()*100:.1f}%)")

## 4. Temporal Train/Test Split

In [ ]:
# Reproduce the same temporal split as nb24
try:
    train_idx = pd.read_pickle(feature_dir / 'train_idx_temporal.pkl')
    test_idx  = pd.read_pickle(feature_dir / 'test_idx_temporal.pkl')
    print("Loaded saved temporal split indices")
except FileNotFoundError:
    # Fall back: load metadata to get years
    df_meta = pd.read_pickle('../data/processed/cleaned_data.pkl')
    year_col = next(c for c in df_meta.columns if 'year' in c.lower())
    years = df_meta.loc[common_idx, year_col]
    train_years = [y for y in years.unique() if y <= 2017]
    train_idx = years[years.isin(train_years)].index
    test_idx  = years[~years.isin(train_years)].index
    print(f"Rebuilt split: train years ≤2017, test years >2017")

train_idx = train_idx.intersection(common_idx)
test_idx  = test_idx.intersection(common_idx)

X_train = X.loc[train_idx]
X_test  = X.loc[test_idx]
y_train = y.loc[train_idx]
y_test  = y.loc[test_idx]

print(f"Train: {X_train.shape}  positive: {y_train.mean()*100:.1f}%")
print(f"Test : {X_test.shape}   positive: {y_test.mean()*100:.1f}%")

## 5. Train and Compare Models

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pos_weight = (1 - y_train.mean()) / y_train.mean()

models = {
    'LR (SPECTER)': LogisticRegression(
        max_iter=3000, solver='saga', class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    'XGBoost (SPECTER)': XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        scale_pos_weight=pos_weight,
        random_state=42, n_jobs=-1, verbosity=0, eval_metric='logloss'
    ),
    'LightGBM (SPECTER)': LGBMClassifier(
        n_estimators=500, num_leaves=31, learning_rate=0.05,
        min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=1.0, class_weight='balanced',
        random_state=42, n_jobs=-1, verbose=-1
    ),
}

results = {}
for name, model in models.items():
    print(f"\nTraining {name}...")
    cv_f1 = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1').mean()
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]

    # Threshold sweep
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.20, 0.65, 0.005):
        f = f1_score(y_test, probs >= t)
        if f > best_f1:
            best_f1, best_t = f, t

    results[name] = {
        'cv_f1': cv_f1,
        'test_f1': best_f1,
        'threshold': best_t,
        'roc_auc': roc_auc_score(y_test, probs)
    }
    print(f"  CV F1={cv_f1:.4f}  Test F1={best_f1:.4f} (t={best_t:.3f})  AUC={results[name]['roc_auc']:.4f}")

In [ ]:
import pandas as pd

baseline_tfidf_f1   = 0.6254
baseline_tfidf_auc  = 0.8104
baseline_xgb_f1     = 0.6188  # nb40b best

print("=" * 65)
print("SPECTER vs TF-IDF COMPARISON")
print("=" * 65)
print(f"{'Model':<30} {'CV F1':>7} {'Test F1':>8} {'AUC':>7} {'vs TF-IDF':>10}")
print("-" * 65)
print(f"{'TF-IDF baseline (LR)':<30} {'—':>7} {baseline_tfidf_f1:>8.4f} {baseline_tfidf_auc:>7.4f} {'baseline':>10}")
print(f"{'XGBoost+FeatureSel (nb40b)':<30} {'—':>7} {baseline_xgb_f1:>8.4f} {'0.8112':>7} {baseline_xgb_f1-baseline_tfidf_f1:>+10.4f}")
print("-" * 65)
for name, r in results.items():
    delta = r['test_f1'] - baseline_tfidf_f1
    print(f"{name:<30} {r['cv_f1']:>7.4f} {r['test_f1']:>8.4f} {r['roc_auc']:>7.4f} {delta:>+10.4f}")

best_name = max(results, key=lambda k: results[k]['test_f1'])
best = results[best_name]
print(f"\n*** BEST: {best_name} — Test F1={best['test_f1']:.4f} ***")

## 6. Save SPECTER Feature Split (for future use)

In [ ]:
X_train.to_pickle(feature_dir / 'X_train_specter.pkl')
X_test.to_pickle(feature_dir / 'X_test_specter.pkl')
y_train.to_pickle(feature_dir / 'y_train_specter.pkl')
y_test.to_pickle(feature_dir / 'y_test_specter.pkl')

print("Saved SPECTER train/test splits.")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")